## Home Sales Price Prediction ? Feature Engineering

This notebook prepares the final modeling dataset following exploratory analysis.

Objectives:

- Define model input features
- Evaluate multicollinearity among numerical variables
- Encode categorical variables
- Construct the final modeling dataset

The resulting dataset will be exported for use in the modeling notebook.

##### Import Libraries and Data

This section imports the libraries required for feature engineering and loads the cleaned dataset generated from the exploratory analysis stage.

Key libraries used:

- **pandas / numpy** ? data manipulation
- **matplotlib / seaborn** ? exploratory diagnostics
- **statsmodels** ? multicollinearity diagnostics (VIF)
- **sklearn** ? encoding and modeling utilities

In [ ]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data Transformations for pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

In [ ]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

In [ ]:
# Load dataset generated from the exploratory analysis notebook
# Parquet format preserves schema and improves load performance
data = pd.read_parquet(CLEAN_DATA_DIR / "02_exploratory_analysis.parquet")
data.head()

In [ ]:
data.shape

### Define Feature Groups

Features are grouped by type to simplify preprocessing and model pipeline construction.

Feature groups:

- **Numerical features**  
  Continuous variables representing property characteristics.

- **Binary features**  
  Indicator variables capturing specific property conditions.

- **Categorical features**  
  Discrete variables representing geographic or structural segmentation.

Separating features by type allows appropriate preprocessing to be applied during modeling.

In [ ]:
# Numerical features
numerical_features = [
    "log_gross_sqft",
    "build_age_yrs", 
    "log_dist_to_station",
    "residential_units"
]

In [ ]:
# Binary features
binary_features = [
    "within_half_mi"
]

In [ ]:
# Categorical features
categorical_features = [
    "neighborhood", 
    "building_class_category"
]

### Multicollinearity Analysis (Variance Inflation Factor)

Highly correlated predictors can destabilize linear regression models and inflate coefficient variance.

To diagnose this, we compute **Variance Inflation Factor (VIF)** for numerical predictors.

Interpretation:

- VIF < 5 ? acceptable
- VIF 5?10 ? moderate correlation
- VIF > 10 ? strong multicollinearity

Variables with excessive VIF are iteratively removed to improve model stability.

In [ ]:
# Import VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Remove VIF Iteratively

def calculate_vif(df, features, thresh):
    X = df[features].copy()
    variables = X.columns.tolist()
    
    while True:
        vif_data = pd.DataFrame()
        vif_data["Feature"] = variables
        vif_data["VIF"] = [
            variance_inflation_factor(X[variables].values, i)
            for i in range(len(variables))
        ]
        
        max_vif = vif_data["VIF"].max()
        
        if max_vif > thresh and len(variables) > 1:
            drop_feature = vif_data.sort_values("VIF", ascending=False)["Feature"].iloc[0]
            print(f"Dropping '{drop_feature}' with VIF: {max_vif:.2f}")
            variables.remove(drop_feature)
        else:
            break
    
    print("\nFinal numeric features:")
    print(variables)
    
    return vif_data.sort_values("VIF", ascending=False)

# Usage
cleaned_df = calculate_vif(data, numerical_features, 10)
cleaned_df

In [ ]:
# Remove "residential_units" and re-run VIF
numerical_features_reduced = [
    "log_gross_sqft",
    "build_age_yrs", 
    "log_dist_to_station"
]

cleaned_df = calculate_vif(data, numerical_features_reduced, 10)
cleaned_df

### Updated Feature Set

Following multicollinearity diagnostics, the numerical feature set was refined to remove highly correlated predictors.

The final feature set combines:

- binary features
- categorical features
- reduced numerical feature set

These features will be used to construct the final modeling matrix.

In [ ]:
# Define features
feature_cols = binary_features + categorical_features + numerical_features_reduced
X = data[feature_cols].copy()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Confirm X is still a DataFrame and has the expected columns
assert hasattr(X, "columns"), "X is not a DataFrame anymore."
print("X columns at transform time:", list(X.columns))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features_reduced),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_processed = preprocessor.fit_transform(X)
X_processed.shape

### Preprocessing Pipeline

Instead of manually encoding categorical variables, a preprocessing pipeline is used.

This ensures that the same transformations applied during training are also applied
during model inference.

The pipeline performs:

? One-hot encoding for categorical variables  
? Feature scaling for numerical variables

In [ ]:
X_processed.shape

### Export Final Modeling Dataset

The fully engineered feature matrix is exported for use in the modeling notebook.

The exported dataset contains:

- encoded predictor variables
- log-transformed target variable

This ensures modeling experiments are reproducible and separated from feature engineering logic.

In [ ]:
X_processed_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index
)

modeling_df = X_processed_df.copy()
modeling_df["log_sale_price"] = data.loc[X.index, "log_sale_price"].values

In [ ]:
modeling_df.to_parquet(CLEAN_DATA_DIR / "03_feature_engineering.parquet", index=False)

### Feature Engineering Summary

The following preprocessing steps were applied to construct the modeling dataset:

1. Feature grouping into numerical, binary, and categorical variables
2. Multicollinearity diagnostics using VIF
3. Removal of highly correlated predictors
4. One-hot encoding of categorical variables
5. Construction of the final modeling matrix

The resulting dataset will be used in the modeling stage to evaluate multiple regression algorithms.

In [ ]:
# Import `joblib` to save prepreprocessing pipeline
import joblib

# Fit preprocessing pipeline
X_processed = preprocessor.fit_transform(X)

joblib.dump(
    preprocessor,
    MODELS_DIR / "preprocessing_pipeline.joblib"
)